# Jersey colour classification in football video


In [6]:
import torch
import pandas as pd
import numpy as np
import os

from dataset import load_manifest
from extract_embeddings import get_embeddings, extract_all_models
from classification_clustering import compare_models, evaluate_single_method, run_clustering

from itertools import product
from IPython.display import display
from extract_embeddings import extract_all_finetuned

from benchmark import run_benchmark, summarize_benchmark, remove_game_from_benchmark

In [7]:
import warnings
warnings.filterwarnings("ignore", module='umap')

device = "cuda" if torch.cuda.is_available() else "cpu"

In [8]:
manifest_path = "dataset_v1/manifest_with_splits.csv"

df = load_manifest(manifest_path)

print("dataset size:", len(df))
df.head()

dataset size: 64291


,crop_path,label,game,src_image,x1,y1,x2,y2,frame_idx,player_id,role_name,left2right,split
0,dataset_v1/crops/game_10_H1__frame_001877__i0_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1399,522,1475,667,1877,101,Left Central Back,1,train
1,dataset_v1/crops/game_10_H1__frame_001877__i1_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1524,290,1591,408,1877,103,Right Central Back,1,train
2,dataset_v1/crops/game_10_H1__frame_001877__i2_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,1091,874,1171,1067,1877,104,Left Back,1,train
3,dataset_v1/crops/game_10_H1__frame_001877__i3_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,77,549,126,704,1877,105,Left Midfielder,1,train
4,dataset_v1/crops/game_10_H1__frame_001877__i4_...,team_left,game_10_H1,/home/anton/Desktop/skoltech/ML/Project/output...,185,169,223,269,1877,106,Right Winger,1,train


In [ ]:
games   = ['game_30', 'game_39', 'game_44', 'game_46']
methods = ["kmeans", "hdbscan", "gmm"]
configs = [
    ("raw",      False, False, False),
    ("umap",     True,  False, False),
    ("umap_pca", True,  True,  False),
]

pretrained_names = ["osnet", "dino"]

finetuned_configs = {
    # "osnet_triplet": ("osnet", "checkpoints/osnet_triplet_best.pth"),
    "osnet_supcon":  ("osnet", "checkpoints/osnet_supcon_best.pth"),
    "dino_supcon":   ("dino",  "checkpoints/dino_supcon_best.pth"),
    # "dino_triplet":  ("dino",  "checkpoints/dino_triplet_best.pth"),
}

all_model_names = pretrained_names + list(finetuned_configs.keys())
total = len(games) * len(all_model_names) * len(methods) * len(configs)
step  = 0
rows  = []


In [ ]:
benchmark_df = run_benchmark(
    games            = games,
    df               = df,
    all_model_names  = all_model_names,
    pretrained_names = pretrained_names,
    finetuned_configs= finetuned_configs,
    methods          = methods,
    configs          = configs,
    device           = device,
    csv_path         = "benchmark.csv",
)


Загружен benchmark.csv: 144 строк, игры: ['game_30', 'game_39', 'game_44', 'game_46']
Все матчи уже посчитаны.


In [ ]:
pd.set_option('display.max_rows', 200)  

benchmark_df

In [12]:
summarize_benchmark(benchmark_df)

=== Полная сводка ===


,model,is_finetuned,method,config,mean_macro_f1,mean_acc,mean_noise
0,osnet_supcon,True,gmm,umap,0.9503,0.9805,NaN
1,osnet_supcon,True,kmeans,umap,0.9503,0.9805,NaN
2,osnet_supcon,True,kmeans,raw,0.9255,0.9655,NaN
3,osnet_supcon,True,kmeans,umap_pca,0.9125,0.9733,NaN
4,osnet_supcon,True,gmm,umap_pca,0.8838,0.9652,NaN
5,osnet_supcon,True,hdbscan,umap_pca,0.8753,0.9163,0.0114
6,dino_supcon,True,hdbscan,umap_pca,0.8674,0.8812,0.0045
7,dino_supcon,True,hdbscan,umap,0.8608,0.9723,0.0031
8,osnet_supcon,True,gmm,raw,0.8484,0.9188,NaN
9,dino,False,gmm,umap_pca,0.8353,0.9298,NaN



=== osnet: pretrained vs finetuned (по mean_macro_f1) ===


,model,method,config,mean_macro_f1,mean_acc
0,osnet_supcon,gmm,umap,0.9503,0.9805
1,osnet_supcon,kmeans,umap,0.9503,0.9805
2,osnet_supcon,kmeans,raw,0.9255,0.9655
3,osnet_supcon,kmeans,umap_pca,0.9125,0.9733
4,osnet_supcon,gmm,umap_pca,0.8838,0.9652
5,osnet_supcon,hdbscan,umap_pca,0.8753,0.9163
8,osnet_supcon,gmm,raw,0.8484,0.9188
10,osnet_supcon,hdbscan,umap,0.8072,0.9711
11,osnet,hdbscan,umap_pca,0.7890,0.8942
18,osnet,hdbscan,umap,0.7212,0.9587



=== dino: pretrained vs finetuned (по mean_macro_f1) ===


,model,method,config,mean_macro_f1,mean_acc
6,dino_supcon,hdbscan,umap_pca,0.8674,0.8812
7,dino_supcon,hdbscan,umap,0.8608,0.9723
9,dino,gmm,umap_pca,0.8353,0.9298
12,dino_supcon,gmm,umap_pca,0.7629,0.8687
13,dino,kmeans,umap_pca,0.7558,0.8815
14,dino_supcon,kmeans,umap_pca,0.7507,0.8768
15,dino_supcon,gmm,umap,0.7297,0.8786
16,dino,hdbscan,umap_pca,0.7262,0.9567
17,dino,hdbscan,umap,0.7224,0.9605
20,dino,gmm,umap,0.6964,0.8433


In [19]:
mask = (
    (benchmark_df["model"] == "osnet_supcon") &
    (benchmark_df["method"].isin(["gmm", "kmeans"])) &
    (benchmark_df["config"] == "umap")
)

display(benchmark_df[mask][[
    "game", "model", "method", "config",
    "clustering_accuracy", "macro_f1_cluster", "noise_fraction"
]].sort_values(["method", "game"]).round(4))

,game,model,method,config,clustering_accuracy,macro_f1_cluster,noise_fraction
25,game_30,osnet_supcon,gmm,umap,0.9808,0.9545,NaN
61,game_39,osnet_supcon,gmm,umap,0.9717,0.8741,NaN
97,game_44,osnet_supcon,gmm,umap,0.9779,0.9782,NaN
133,game_46,osnet_supcon,gmm,umap,0.9916,0.9943,NaN
19,game_30,osnet_supcon,kmeans,umap,0.9808,0.9545,NaN
55,game_39,osnet_supcon,kmeans,umap,0.9717,0.8741,NaN
91,game_44,osnet_supcon,kmeans,umap,0.9779,0.9782,NaN
127,game_46,osnet_supcon,kmeans,umap,0.9916,0.9943,NaN
